# 2.1 Train pipeline — cashback + loyalty connection + dynamic FS


In [1]:
# Версия обучающих данных
DATASET_POSTFIX = "nbc_monthly_202610_rub_filtered"

# Параметры периода обучения
TRAIN_START = "2025-09-01"
TRAIN_END_EXCLUSIVE = "2026-10-01"
VALID_MONTH = "2026-10-01"
TEST_MONTH = None

# Период применения модели
INFERENCE_MONTH = "2026-12-01"

In [2]:
# Настройка рабочей директории
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta
import sys
import os
import json

_branch = "dynamic"
_cwd = Path.cwd().resolve()

_candidates = [_cwd, _cwd / _branch, _cwd.parent / _branch]

PROJECT_ROOT = next(
    (
        x for x in _candidates
        if x.name == _branch
        and (x / "srcc" / "nbc_project").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найдена папка {_branch}/srcc/nbc_project"
    )

SRC_PATH = PROJECT_ROOT / "srcc"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)

# Название эксперимента
EXPERIMENT_NAME = "202610_dynamic_rub_filtered"

# Формирование идентификатора запуска
train_start_tag = datetime.strptime(
    TRAIN_START, "%Y-%m-%d"
).strftime("%Y%m")

train_end_tag = (
    datetime.strptime(TRAIN_END_EXCLUSIVE, "%Y-%m-%d")
    - relativedelta(months=1)
).strftime("%Y%m")

valid_tag = datetime.strptime(
    VALID_MONTH, "%Y-%m-%d"
).strftime("%Y%m")

test_tag = (
    datetime.strptime(TEST_MONTH, "%Y-%m-%d").strftime("%Y%m")
    if TEST_MONTH is not None
    else "none"
)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

RUN_TAG = (
    f"train_{train_start_tag}_{train_end_tag}"
    f"__valid_{valid_tag}"
    f"__test_{test_tag}"
    f"__{run_timestamp}"
)

# Директории конкретного запуска
RUN_DIR = (
    PROJECT_ROOT
    / "experiment_outputs"
    / EXPERIMENT_NAME
    / RUN_TAG
)

MODEL_DIR = str(RUN_DIR / "models" / "train_pipeline")

REPORT_DIR = RUN_DIR / "reports"

FS_REPORT_DIR = str(REPORT_DIR / "feature_selection")
VISUALS_REPORT_DIR = str(REPORT_DIR / "training_visuals")
ROLLING_REPORT_DIR = str(REPORT_DIR / "rolling_backtest")

# Создание директорий
RUN_DIR.mkdir(parents=True, exist_ok=False)

for directory in [
    MODEL_DIR,
    FS_REPORT_DIR,
    VISUALS_REPORT_DIR,
    ROLLING_REPORT_DIR
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

# Сохранение конфигурации запуска
run_config = {
    "experiment_name": EXPERIMENT_NAME,
    "run_tag": RUN_TAG,
    "dataset_postfix": DATASET_POSTFIX,
    "train_start": TRAIN_START,
    "train_end_exclusive": TRAIN_END_EXCLUSIVE,
    "valid_month": VALID_MONTH,
    "test_month": TEST_MONTH,
    "inference_month": INFERENCE_MONTH,
    "run_timestamp": run_timestamp
}

with open(RUN_DIR / "run_config.json", "w", encoding="utf-8") as file:
    json.dump(run_config, file, indent=4, ensure_ascii=False)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("RUN_TAG:", RUN_TAG)
print("RUN_DIR:", RUN_DIR)
print("MODEL_DIR:", MODEL_DIR)

PROJECT_ROOT: /home/jovyan/work/loyalty_handoff/dynamic
EXPERIMENT_NAME: 202610_dynamic_rub_filtered
RUN_TAG: train_202509_202609__valid_202610__test_none__20261122_075913_858880
RUN_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880
MODEL_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/train_pipeline


In [3]:
import pandas as pd

from nbc_project.train_pipeline_workflow import (
    train_cashback_connected_clean,
    train_loyalty_connection_clean,
)
from nbc_project.training_visuals import (
    make_regression_report,
    make_classification_report,
    plot_rolling_backtest_summary,
)
from nbc_project.rolling_backtest import run_rolling_backtest
from nbc_project.feature_lists import get_feature_list

import gc

In [4]:
# Пути к ранее скачанным обучающим датасетам
COMMON_DATA_DIR = PROJECT_ROOT.parent / "common" / "data"

DATA_CASHBACK_FULL = str(
    COMMON_DATA_DIR
    / f"cb_train_modular_full_connected_{DATASET_POSTFIX}"
)

DATA_LOYAL_FLG_FULL = str(
    COMMON_DATA_DIR
    / f"cb_sample_for_binary_modular_full_{DATASET_POSTFIX}"
)

# Проверка наличия датасетов
for name, path in [
    ("Cashback", DATA_CASHBACK_FULL),
    ("Loyalty connection", DATA_LOYAL_FLG_FULL)
]:
    assert Path(path).is_dir(), f"Датасет не найден: {path}"
    print(f"{name}: {path}")

Cashback: /home/jovyan/work/loyalty_handoff/common/data/cb_train_modular_full_connected_nbc_monthly_202610_rub_filtered
Loyalty connection: /home/jovyan/work/loyalty_handoff/common/data/cb_sample_for_binary_modular_full_nbc_monthly_202610_rub_filtered


In [5]:
USE_DYNAMIC_FS = True

# Финальная конфигурация после benchmark и переноса выбранных подходов
# на модели, для которых отдельный benchmark не прогоняли.
FS_BY_MODEL = {
    "cashback_connected": "psi_importance_auto_n",
    "loyalty_connect": "psi_importance_auto_n"
}

DYNAMIC_FS_PRESETS = {
    "psi_importance_auto_n": {
        "method": "psi_importance",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 200,
        "report_dir": FS_REPORT_DIR,
    },
    "psi_rfe_loss_auto_n": {
        "method": "psi_rfe",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "rfe_algorithm": "loss_function_change",
        "rfe_steps": 5,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 180,
        "report_dir": FS_REPORT_DIR,
    },
}

CORRELATION_PREFILTER = {
    "drop_correlated_features": True,
    "correlation_threshold": 0.95,
    "correlation_method": "pearson",
    "correlation_max_rows": 20_000,
    "correlation_min_unique_values": 20,
}

for _cfg in DYNAMIC_FS_PRESETS.values():
    _cfg.update(CORRELATION_PREFILTER)

PROTECTED_OFFER_FEATURES = [
    c
    for c in get_feature_list("data_fm_must_have_features")
    if c.startswith("offered_percent_cat_")
]

if not PROTECTED_OFFER_FEATURES:
    PROTECTED_OFFER_FEATURES = [
        c
        for c in get_feature_list("must_have_offered_percent_features")
        if c.startswith("offered_percent_cat_")
    ]

def make_fs_config(model_key):
    preset = FS_BY_MODEL[model_key]
    cfg = DYNAMIC_FS_PRESETS[preset].copy()
    cfg["protected_features"] = PROTECTED_OFFER_FEATURES.copy()
    return cfg

print("FINAL FS_BY_MODEL:", FS_BY_MODEL)
print("protected offered_percent:", len(PROTECTED_OFFER_FEATURES))
print("FS_REPORT_DIR:", FS_REPORT_DIR)


def fs_for(model_key):
    return make_fs_config(model_key)

RUN_ROLLING_BACKTEST = False

FINAL FS_BY_MODEL: {'cashback_connected': 'psi_importance_auto_n', 'loyalty_connect': 'psi_importance_auto_n'}
protected offered_percent: 41
FS_REPORT_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/feature_selection


## Параметры периода

In [6]:
# Проверка корректности периодов
assert TRAIN_START < TRAIN_END_EXCLUSIVE
assert TRAIN_END_EXCLUSIVE <= VALID_MONTH

if TEST_MONTH is not None:
    assert VALID_MONTH < TEST_MONTH

# Проверка, что даты не изменились после создания RUN_DIR
assert TRAIN_START == run_config["train_start"]
assert TRAIN_END_EXCLUSIVE == run_config["train_end_exclusive"]
assert VALID_MONTH == run_config["valid_month"]
assert TEST_MONTH == run_config["test_month"]

print("TRAIN_START:", TRAIN_START)
print("TRAIN_END_EXCLUSIVE:", TRAIN_END_EXCLUSIVE)
print("VALID_MONTH:", VALID_MONTH)
print("TEST_MONTH:", TEST_MONTH)
print("INFERENCE_MONTH:", INFERENCE_MONTH)
print("MODEL_DIR:", MODEL_DIR)

TRAIN_START: 2025-09-01
TRAIN_END_EXCLUSIVE: 2026-10-01
VALID_MONTH: 2026-10-01
TEST_MONTH: None
INFERENCE_MONTH: 2026-12-01
MODEL_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/train_pipeline


## 1. Регрессия cashback для подключённых клиентов

In [7]:
%%time

from pathlib import Path

assert Path(DATA_CASHBACK_FULL).is_dir(), "Не найдена cashback-выборка"

# Защита от перезаписи обученной модели
assert not (Path(MODEL_DIR) / "cashback_connected.cbm").exists(), (
    "Модель уже существует в директории текущего запуска"
)

cashback_result = train_cashback_connected_clean(
    data_path=DATA_CASHBACK_FULL,
    train_start=TRAIN_START,
    train_end_exclusive=TRAIN_END_EXCLUSIVE,
    valid_month=VALID_MONTH,
    test_month=TEST_MONTH,
    model_dir=MODEL_DIR,
    feature_selection=fs_for("cashback_connected"),
    model_key="cashback_connected",
    model_filename="cashback_connected.cbm",
)

print("cashback features:", len(cashback_result["feature_cols"]))

cashback_result["metrics"]

dynamic feature selection: method=psi_importance, sample_rows=150,000, psi_threshold=0.2
dynamic ignore cols: 14
selected features: 227 / 1353
train shape: (1557688, 245)
valid shape: (123429, 245)
test shape: (0, 245)
0:	learn: 716.1242366	test: 665.5662425	best: 665.5662425 (0)	total: 978ms	remaining: 32m 35s
100:	learn: 594.7186636	test: 565.6849053	best: 565.6849053 (100)	total: 1m 16s	remaining: 24m 4s
200:	learn: 587.2272008	test: 562.1846250	best: 562.1846250 (200)	total: 2m 30s	remaining: 22m 31s
300:	learn: 584.1837035	test: 560.9214539	best: 560.9214539 (300)	total: 3m 45s	remaining: 21m 10s
400:	learn: 582.1398012	test: 559.6774090	best: 559.6774090 (400)	total: 5m	remaining: 19m 59s
500:	learn: 580.5669212	test: 558.4592884	best: 558.4592884 (500)	total: 6m 15s	remaining: 18m 42s
600:	learn: 579.1746991	test: 557.7086397	best: 557.7077305 (599)	total: 7m 29s	remaining: 17m 26s
700:	learn: 578.0029553	test: 557.0099620	best: 556.9887012 (698)	total: 8m 44s	remaining: 16m 11s

{'valid': {'rows': 123429,
  'mae': 280.5767078360078,
  'rmse': 554.1742301215849,
  'median_ae': 147.24269255802545,
  'r2': 0.31324063300031846,
  'wape': 0.9059153832953238}}

In [8]:
split_targets = {
    "train": cashback_result["train"]["target_cashback_month"],
    "valid": cashback_result["valid"]["target_cashback_month"],
}

if not cashback_result["test"].empty:
    split_targets["test"] = cashback_result["test"]["target_cashback_month"]
    y_true_cashback = cashback_result["test"]["target_cashback_month"]
    y_pred_cashback = cashback_result["pred_test"]
    cashback_report_suffix = "test_oot"
else:
    y_true_cashback = cashback_result["valid"]["target_cashback_month"]
    y_pred_cashback = cashback_result["pred_valid"]
    cashback_report_suffix = "valid"

cashback_visuals = make_regression_report(
    model_key=f"cashback_connected_{cashback_report_suffix}",
    target_col="target_cashback_month",
    y_true=y_true_cashback,
    y_pred=y_pred_cashback,
    output_dir=VISUALS_REPORT_DIR,
    split_targets=split_targets,
    model=cashback_result["model"],
)

print("Отчёты сохранены:", VISUALS_REPORT_DIR)

Интерактивные Plotly-графики сохранены в: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals/cashback_connected_valid
Отчёты сохранены: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals


In [9]:
del cashback_result, cashback_visuals
gc.collect()

5841

### Dynamic FS для loyalty: отдельный calibration holdout

При `USE_DYNAMIC_FS=True` feature selection использует только `train + valid_model`. `valid_calib` не участвует в PSI / RFE / permutation / null importance и остаётся отдельной половиной месяца для isotonic calibration и проверки качества.


## 2. Бинарная модель подключения loyalty

In [10]:
%%time

# Проверка источника данных
assert Path(DATA_LOYAL_FLG_FULL).is_dir(), (
    f"Не найден датасет: {DATA_LOYAL_FLG_FULL}"
)

# Защита от перезаписи результатов обучения
assert not (Path(MODEL_DIR) / "loyalty_connect.cbm").exists(), (
    "Модель loyalty_connect уже существует в текущем запуске"
)

assert not (Path(MODEL_DIR) / "isotonic_calibrator_loyalty.pkl").exists(), (
    "Калибратор loyalty_connect уже существует в текущем запуске"
)

loyalty_result = train_loyalty_connection_clean(
    data_path=DATA_LOYAL_FLG_FULL,
    train_start=TRAIN_START,
    train_end_exclusive=TRAIN_END_EXCLUSIVE,
    valid_month=VALID_MONTH,
    test_month=TEST_MONTH,
    model_dir=MODEL_DIR,
    feature_selection=fs_for("loyalty_connect"),
    model_key="loyalty_connect",
    model_filename="loyalty_connect.cbm",
    calibrator_filename="isotonic_calibrator_loyalty.pkl",
)

print("loyalty features:", len(loyalty_result["feature_cols"]))

loyalty_result["metrics"]

dynamic feature selection: method=psi_importance, sample_rows=150,000, psi_threshold=0.2
dynamic ignore cols: 15
selected features: 182 / 1353
train shape: (3250000, 198)
valid_full shape: (250000, 198)
valid_model shape: (125000, 198)
valid_calib shape: (125000, 198)
test shape: (0, 198)
0:	test: 0.9152728	best: 0.9152728 (0)	total: 2.41s	remaining: 1h 20m 16s
100:	test: 0.9370605	best: 0.9370605 (100)	total: 3m 16s	remaining: 1h 1m 38s
200:	test: 0.9409866	best: 0.9409866 (200)	total: 6m 38s	remaining: 59m 30s
300:	test: 0.9428291	best: 0.9428291 (300)	total: 9m 59s	remaining: 56m 26s
400:	test: 0.9439455	best: 0.9439455 (400)	total: 13m 21s	remaining: 53m 15s
500:	test: 0.9448103	best: 0.9448103 (500)	total: 16m 42s	remaining: 50m
600:	test: 0.9456020	best: 0.9456020 (600)	total: 20m 2s	remaining: 46m 39s
700:	test: 0.9462930	best: 0.9462930 (700)	total: 23m 18s	remaining: 43m 11s
800:	test: 0.9468305	best: 0.9468305 (800)	total: 26m 28s	remaining: 39m 38s
900:	test: 0.9472756	best:

{'valid_model_raw': {'rows': 125000,
  'positive_rate': 0.087272,
  'avg_score': 0.23282260699811044,
  'precision@0.5': 0.45857307249712315,
  'recall@0.5': 0.8767073058942157,
  'f1@0.5': 0.6021722021092397,
  'roc_auc': 0.9497107037323933,
  'pr_auc': 0.744865187216577,
  'logloss': 0.27773622296623923,
  'brier': 0.0793135740848426,
  'tn': 102799,
  'fp': 11292,
  'fn': 1345,
  'tp': 9564},
 'valid_calib_raw': {'rows': 125000,
  'positive_rate': 0.08744,
  'avg_score': 0.23266049927103508,
  'precision@0.5': 0.45775359876751237,
  'recall@0.5': 0.8698993595608417,
  'f1@0.5': 0.5998548941673765,
  'roc_auc': 0.9481210414689749,
  'pr_auc': 0.7436358150911058,
  'logloss': 0.27766337877204367,
  'brier': 0.07937768116728132,
  'tn': 102807,
  'fp': 11263,
  'fn': 1422,
  'tp': 9508},
 'valid_calib': {'rows': 125000,
  'positive_rate': 0.08744,
  'avg_score': 0.08743999999999999,
  'precision@0.5': 0.7732346517835477,
  'recall@0.5': 0.5830741079597438,
  'f1@0.5': 0.664823701230961

In [11]:
if not loyalty_result["test"].empty:
    y_true = loyalty_result["test"]["target_loyalty_connected_flg"]
    y_score = loyalty_result["pred_test"]
    report_suffix = "test"
else:
    y_true = loyalty_result["valid_calib"]["target_loyalty_connected_flg"]
    y_score = loyalty_result["pred_valid_calib"]
    report_suffix = "valid_calib"

split_targets = {
    "train": loyalty_result["train"]["target_loyalty_connected_flg"],
    "valid_model": loyalty_result["valid_model"]["target_loyalty_connected_flg"],
    "valid_calib": loyalty_result["valid_calib"]["target_loyalty_connected_flg"],
}

if not loyalty_result["test"].empty:
    split_targets["test"] = loyalty_result["test"]["target_loyalty_connected_flg"]

loyalty_visuals = make_classification_report(
    model_key=f"loyalty_connect_{report_suffix}",
    target_col="target_loyalty_connected_flg",
    y_true=y_true,
    y_score=y_score,
    output_dir=VISUALS_REPORT_DIR,
    split_targets=split_targets,
    model=loyalty_result["model"],
    threshold=0.5,
)

print("Отчёты сохранены:", VISUALS_REPORT_DIR)

Интерактивные Plotly-графики сохранены в: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals/loyalty_connect_valid_calib
Отчёты сохранены: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals


In [12]:
del loyalty_result
gc.collect()

4718